In [1]:
from importlib import reload   # recharge automatiquement les MàJ du module.
import Conn_db
import Utils as ul
import Matching

import warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")  # supprime le warning SQL Alchemy qu'est inutile

reload(Conn_db)
reload(ul)
reload(Matching)

Looking in indexes: http://10.4.3.215:8081/repository/pypi-all/simple
Looking in indexes: http://10.4.3.215:8081/repository/pypi-all/simple
Looking in indexes: http://10.4.3.215:8081/repository/pypi-all/simple
Looking in indexes: http://10.4.3.215:8081/repository/pypi-all/simple
Looking in indexes: http://10.4.3.215:8081/repository/pypi-all/simple
Looking in indexes: http://10.4.3.215:8081/repository/pypi-all/simple
Looking in indexes: http://10.4.3.215:8081/repository/pypi-all/simple
Looking in indexes: http://10.4.3.215:8081/repository/pypi-all/simple
Looking in indexes: http://10.4.3.215:8081/repository/pypi-all/simple
Looking in indexes: http://10.4.3.215:8081/repository/pypi-all/simple
Looking in indexes: http://10.4.3.215:8081/repository/pypi-all/simple
Looking in indexes: http://10.4.3.215:8081/repository/pypi-all/simple
Looking in indexes: http://10.4.3.215:8081/repository/pypi-all/simple
Looking in indexes: http://10.4.3.215:8081/repository/pypi-all/simple


<module 'Matching' from '/home/jovyan/work/sirenisation_finess/Matching.py'>

In [2]:
from Conn_db import get_finess_active, get_sirene_active, count_finess_active
import Utils as ul
from Matching import run_direct_matching, run_indirect_matching_by_department, merge_phase2_parquets_to_excel
import pandas as pd
import os
from datetime import datetime

## Pase 1: matching directe

In [3]:

# ======================
# Phase 1
# ======================

# 1. Se connecter aux bases de données
# ====================================
conn_sql = get_finess_active()
con_duck = get_sirene_active(read_only=True)


# 2. Lancer le matching (phase 1) 
# ================================================
%time valid_df, rejected_df = run_direct_matching(conn_sql, con_duck, threshold= 67, export_excel=True)


✅ Connexion SQL Server réussie
✅ Connexion DuckDB (Lecture) : /home/jovyan/work/sirenisation_finess/sirene.duckdb
Phase 1 - Matching direct
1. Chargement des FINESS avec SIREN...
Nombre de Finess ayant un num_siren : 52020
Nombre de SIREN distincts : 51413
2. Préprocessing...
3. Chargement des références SIRENE...
4. Matching direct...
Threshold utilisé : 67
Matching terminé
Validés : 38665
Rejetés : 13355
Total lignes : 52020
Excel généré : matching_phase1.xlsx
CPU times: user 1min 15s, sys: 995 ms, total: 1min 16s
Wall time: 1min 15s


In [4]:
print(f'{valid_df['siren'].nunique()}')


38528


## Pase 2: matching indirecte

In [5]:
"""
# Génerer les parquets siren par dep 
SIRENE_PARQUET_DIR = "data/sirene_parquets"

con_duck = get_sirene_active(read_only=True)
ul.build_sirene_dep(con_duck, SIRENE_PARQUET_DIR)
"""

'\n# Génerer les parquets siren par dep \nSIRENE_PARQUET_DIR = "data/sirene_parquets"\n\ncon_duck = get_sirene_active(read_only=True)\nul.build_sirene_dep(con_duck, SIRENE_PARQUET_DIR)\n'

In [7]:

# =========================
# Phase 2
# =========================
conn_sql = get_finess_active()
con_duck = get_sirene_active(read_only=True)

OUTPUT_DIR = "phase2_results_by_dep"
CHECKPOINT_FILE = f"{OUTPUT_DIR}/departments_done.txt"

os.makedirs(OUTPUT_DIR, exist_ok=True)    # exist_ok = évite le crash si le dossier crée déjà 

print("Checkpoint existant ?", os.path.exists(CHECKPOINT_FILE))


# 1. Chargement SIRENE (parquet par département)
# ===============================================

SIRENE_PARQUET_DIR = "/home/jovyan/work/sirenisation_finess/data/sirene_parquets"
sirene_loader = ul.SireneLoader(SIRENE_PARQUET_DIR)


# 2. Lancer le matching + Blocking par dèpartement (phase 2)
# ==========================================================
import time

start = time.perf_counter()

run_indirect_matching_by_department(
    conn_sql=conn_sql,
    valid_df=valid_df,
    sirene_loader=sirene_loader,
    output_dir=OUTPUT_DIR,
    checkpoint_file=f"{OUTPUT_DIR}/departments_done.txt",
    chunk_size=5000,
)

elapsed = time.perf_counter() - start
print(f"Temps d'exécution : {elapsed:.2f} secondes")


# 3. Merge + export Excel final
# ==============================
df_phase2 = merge_phase2_parquets_to_excel(
    input_dir=OUTPUT_DIR,
    output_excel="matching_phase2_final.xlsx"
)

print("Phase 2 terminée avec succès")


✅ Connexion SQL Server réussie
✅ Connexion DuckDB (Lecture) : /home/jovyan/work/sirenisation_finess/sirene.duckdb
Checkpoint existant ? True
Phase 2 - Matching indirect par département
Départements déjà traités : {'45', '11', '01', '93', '58', '07', '73', '86', '49', '2A', '59', '77', '04', '72', '83', '65', '34', '92', '64', '60', '48', '14', '38', '66', '35', '39', '75', '62', '90', '51', '88', '25', '53', '55', '08', '42', '79', '02', '69', '16', '41', '82', '46', '2B', '54', '27', '18', '85', '89', '21', '26', '22', '74', '67', '76', '81', '09', '36', '44', '56', '32', '17', '30', '91', '37', '43', '40', '13', '84', '24', '52', '71', '19', '06', '57', '68', '78', '80', '61', '33', '47', '50', '05', '70', '31', '03', '12', '63', '15', '28', '23', '10', '29', '87'}
Département 01 déjà traité → skip
Département 02 déjà traité → skip
Département 03 déjà traité → skip
Département 04 déjà traité → skip
Département 05 déjà traité → skip
Département 06 déjà traité → skip
Département 07 déj

Dep 94 - chunk 0:   0%|          | 0/291 [00:00<?, ?it/s]

Résultats écrits : phase2_results_by_dep/phase2_dep_94_part_0.parquet (873 lignes)
Département 94 enregistré comme terminé (873 résultats)

===== Traitement du département 95 =====
Dep 95 - chunk 0 - 307 lignes


Dep 95 - chunk 0:   0%|          | 0/307 [00:00<?, ?it/s]

Résultats écrits : phase2_results_by_dep/phase2_dep_95_part_0.parquet (918 lignes)
Département 95 enregistré comme terminé (918 résultats)

===== Traitement du département 97 =====
Dep 97 - chunk 0 - 702 lignes


Dep 97 - chunk 0:   0%|          | 0/702 [00:00<?, ?it/s]

Résultats écrits : phase2_results_by_dep/phase2_dep_97_part_0.parquet (2106 lignes)
Département 97 enregistré comme terminé (2106 résultats)
Phase 2 terminée
Temps d'exécution : 2665.42 secondes
97 fichiers parquet détectés
Total lignes : 46329
Excel généré : matching_phase2_final.xlsx
Structures avec bon candidat : 6815
Structures sans aucun bon candidat : 8628
Phase 2 terminée avec succès
